<a href="https://colab.research.google.com/github/AdithyaAnand24/SIMC-2026/blob/main/src/task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 3

In [ ]:
#Module purpose. Question 3a, part (i): detect and remove the corrupted "empty vial" measurements from the spectroscopy design matrix mystery_matrix1.npy.

'''Data contract / assumptions.
  - The file loads as a float64 array of shape (3401, 1000). The problem defines the design matrix X with samples as rows (N = 1000 vials) and features as
  columns (frequencies). The on-disk array is therefore transposed relative to that convention: its 1000 axis is the samples. We transpose once, up front,
  so that downstream code follows the report's row = sample convention. (Note: the problem text states M = 3041 features; the actual file has 3401 columns.
  We use the array's true dimension and flag the discrepancy rather than hard-coding either number.)
  - All entries are non-negative intensities in [0, ~0.85]; there are no NaN/inf values. This is asserted, not assumed silently.

  Detection method. Each measurement is reduced to a single scalar, its row norm ‖xᵢ‖₂ = total signal energy across all frequencies. A real sample has
  absorption peaks (high norm); an empty vial is low background noise everywhere (low norm). The norm distribution is bimodal with an empty gap, so a hard
  threshold placed anywhere inside the gap separates the two populations exactly.

  Threshold choice & robustness. The cutoff is 3.0, but the result is invariant to any threshold in [2.0, 4.0] because the gap is genuinely empty. This
  invariance is checked explicitly — it is the evidence that the cut is principled rather than arbitrary.

  Validation strategy. Two independent checks: (1) summary statistics (mean/max/per-channel spread) must show the removed group is flat and low while the
  kept group has structure; (2) a visual overlay of removed vs kept spectra must show flat noise vs clear peaks.

  Outputs. T (number removed, expected 40), a boolean mask empty, the cleaned matrix S_clean (960 × 3401), and a figure q3a_part_i.png. The cleaned matrix
  is the input to 3b onward.'''



In [ ]:
#Cell 1 — Imports

import os
import numpy as np
import matplotlib.pyplot as plt

# Run from the repository root so the relative data/ and data/output/TASK 3/ paths
# below resolve, whether this notebook is launched from src/ or the repo root.
for _ in range(5):
    if os.path.exists("data/input/mystery_matrix1.npy"):
        break
    os.chdir("..")
DATA_PATH = "data/input/mystery_matrix1.npy"


In [ ]:
#Cell 2 — Load and sanity-check the raw array

# Load the raw design-matrix file.
# NOTE: the file is stored as (3401, 1000). The problem convention is
# samples-as-rows with N = 1000 vials, so the file is effectively X^T.
raw = np.load(DATA_PATH)

print("raw shape :", raw.shape)
print("dtype     :", raw.dtype)
print("min / max :", raw.min(), raw.max())
print("NaN / inf :", np.isnan(raw).any(), np.isinf(raw).any())

# The problem states M = 3041 features; the file has 3401. We trust the file
# and key off the axis whose length is 1000 (the samples).
assert 1000 in raw.shape, "expected one axis to be the 1000 samples"



In [ ]:
#Cell 3 — Orient to "samples as rows"

# Transpose if necessary so that rows = samples (vials), cols = frequencies.
S = raw.T if raw.shape[0] != 1000 else raw
N, M = S.shape
print(f"design matrix oriented: {N} samples (rows) x {M} frequencies (cols)")



In [ ]:
#Cell 4 — Reduce each measurement to its row norm

# Row norm = total signal energy of one vial across all frequencies.
# Empty vials (background noise only) have small norm; real samples have peaks.
norms = np.linalg.norm(S, axis=1)
print("row-norm summary")
print("  min   :", norms.min())
print("  max   :", norms.max())
print("  mean  :", norms.mean())
print("  median:", np.median(norms))



In [ ]:
#Cell 5 — Inspect the distribution (find the gap)
# A text histogram makes the bimodal structure and the empty gap obvious.
hist, edges = np.histogram(norms, bins=40)
for h, e in zip(hist, edges[:-1]):
  print(f"  {e:5.2f} : {'#' * h} {h}")
# A tight clump of 40 sits at ~1.58-1.76, then an empty gap until ~4.15.



In [ ]:
#Cell 6 — Threshold and confirm robustness
#The empty-vial clump is well below the real samples. Any threshold inside the
#gap gives the same count, which is the evidence the cut is not arbitrary.
for thr in (2.0, 3.0, 4.0):
    print(f"  rows with norm < {thr}: {(norms < thr).sum()}")  # all 40

THRESHOLD = 3.0
empty = norms < THRESHOLD          # boolean mask of corrupted measurements
real  = ~empty

T = int(empty.sum())
print(f"\nT = {T} corrupted (empty-vial) measurements")
print("their norm range:", norms[empty].min(), "->", norms[empty].max())
print("remaining measurements:", int(real.sum()))   # 960



In [ ]:
#Cell 7 — Validate the removal (statistics)
# Removed group should be flat & low; kept group should have structure (peaks).
def describe(mask, label):
  print(f"{label:12s}: mean/chan = {S[mask].mean():.4f}  "
      f"max = {S[mask].max():.4f}  "
      f"per-channel std = {S[mask].std(axis=1).mean():.4f}")
describe(empty, "empty (rm)")
describe(real,  "real (keep)")



In [ ]:
#Cell 8 — Validate the removal (figure)

# Single panel: the row-norm distribution with the empty-vial threshold marked.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(norms, bins=60, color="steelblue", edgecolor="k", linewidth=0.3)
ax.axvline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
ax.set_xlabel(r"row norm  $\|x_i\|$")
ax.set_ylabel("count")
ax.set_title(f"Row-norm distribution (1000 measurements)\n"
             f"{T} empty vials isolated below the gap")
ax.legend()
plt.tight_layout()
import os
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_i.png", dpi=120)
plt.show()

In [ ]:
#Cell 9 — Produce the cleaned matrix for downstream parts

  # Cleaned design matrix after removing the T empty-vial measurements.
  # This (960 x 3401) array is the input to 3a(ii) and 3b onward.
S_clean = S[real]
print("cleaned design matrix:", S_clean.shape)   # (960, 3401)

In [ ]:
#Cell 10 — Question 3a, part (ii): the other error (high-norm outliers)

# Besides the empty vials, the row-norm distribution shows a SECOND group far
# above the bulk, separated by a clear gap (~5x wider than any gap inside the
# bulk). These measurements form a contiguous block (rows 924-959) sitting
# directly in front of the empty vials (rows 960-999): the whole tail of the
# matrix is anomalous, so we flag them as the "other error".

sn   = np.sort(norms)
gaps = np.diff(sn)
mid  = len(sn) // 2
hi   = mid + int(np.argmax(gaps[mid:]))        # widest gap ABOVE the median
high_thr = 0.5 * (sn[hi] + sn[hi + 1])

high = norms > high_thr                         # mask: high-norm outliers
print(f"high-norm threshold : {high_thr:.3f}")
print(f"outliers found      : {int(high.sum())}  "
      f"(rows {np.where(high)[0].min()}-{np.where(high)[0].max()})")
print(f"norm range          : {norms[high].min():.2f}-{norms[high].max():.2f}"
      f"  (bulk median {np.median(norms):.2f})")

In [ ]:
#Cell 11 — Produce the fully cleaned matrix (both errors removed)

# Keep a vial only if it is neither an empty blank nor a high-norm outlier.
keep = real & ~high
S_clean = S[keep]                               # overwrite: this is now final

print("removed - empty vials    :", int(empty.sum()))   # 40
print("removed - high-norm group:", int(high.sum()))    # 36
print("cleaned design matrix    :", S_clean.shape)       # (924, 3401)

In [ ]:
#Cell 12 — Evidence: are the 36 high-norm vials their own cluster?

# Concrete test of the part-(ii) decision. Cluster the empties-removed data in
# PCA space WITH vs WITHOUT the 36 high-norm vials. If they are a single
# distinct group, removing them should drop the cluster count by exactly one
# and leave every other cluster untouched.
import os

def pca_scores(X, k=4):
    # project samples onto their top-k principal components (via the Gram matrix)
    Xc = X - X.mean(0)
    w, v = np.linalg.eigh(Xc @ Xc.T)
    idx = np.argsort(w)[::-1][:k]
    return v[:, idx] * np.sqrt(np.maximum(w[idx], 0.0))

def count_clusters(P, r):
    # leader clustering (numpy-only): start a new cluster when a point is
    # farther than r from every existing cluster centre
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

for label, mask in [("WITH    high-norm (960 vials)", real),
                    ("WITHOUT high-norm (924 vials)", real & ~high)]:
    P = pca_scores(S[mask], k=4)
    D = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    r = 8 * np.median(D.min(1))                  # radius ~ 8x typical neighbour spacing
    print(f"{label}: {count_clusters(P, r)} clusters")

# How isolated is the group, in the top-4 PC space?
P = pca_scores(S[real], k=4)
is_high = high[real]                             # which of the 960 kept vials are high-norm
ch     = P[is_high].mean(0)
radius = np.sqrt(((P[is_high]  - ch) ** 2).sum(1)).mean()
gap    = np.sqrt(((P[~is_high] - ch) ** 2).sum(1)).min()
print(f"\nhigh-norm cluster radius = {radius:.2f}, gap to nearest other vial = {gap:.2f} "
      f"({gap / radius:.0f}x its own radius)")
print("=> removing the 36 deletes exactly one well-separated cluster; the rest are untouched.")

# Visual: top-2 PCs with the 36 highlighted
plt.figure(figsize=(6, 5))
plt.scatter(P[~is_high, 0], P[~is_high, 1], s=10, alpha=0.4, label="other vials")
plt.scatter(P[is_high, 0],  P[is_high, 1],  s=30, color="red", label="high-norm group (36)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("The 36 high-norm vials form one isolated cluster")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_ii_clusters.png", dpi=120)
plt.show()

In [ ]:
#Cell 13 — Question 3b: mean-centre and per-feature variance

# Mean-centre each feature (column) so it has zero average across the N~ vials.
N_tilde = S_clean.shape[0]
X_centered = S_clean - S_clean.mean(axis=0)          # X~, shape (924, 3401)

# Covariance matrix C = (1/N~) X~^T X~  (M x M).  We only need its DIAGONAL here:
# C_jj = variance of feature j.  np.var divides by N~, matching the definition.
variances = X_centered.var(axis=0)                   # = diag(C)

# Sanity check: trace(C) = total variance (invariant under rotation/compression).
print("total variance  tr(C) =", round(variances.sum(), 4))

# 3b(i): the single feature with the largest variance
j_star = int(np.argmax(variances))
print(f"3b(i)  largest-variance feature  j* = {j_star}  "
      f"(variance = {variances[j_star]:.5f})")

In [ ]:
#Cell 14 — Question 3b(ii): plot the variance as a function of feature index

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(variances, lw=0.6, color="steelblue")
ax.axvline(j_star, color="red", ls="--", lw=1, label=f"j* = {j_star}  (largest-variance feature)")
ax.set_xlabel("feature index j  (frequency channel)")
ax.set_ylabel(r"variance  $C_{jj}$")
ax.set_title("Per-feature variance across the 3401 frequencies")
ax.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3b_variance.png", dpi=120)
plt.show()

**3b(iii) — what the variance distribution reveals.**

The variance is *not* spread evenly across the 3401 frequencies. It is concentrated in a
handful of narrow bands (peaks) separated by broad, low-variance baseline regions — the
variance profile itself looks like a spectrum. Those high-variance bands are exactly the
frequencies where the dominant molecules absorb and where the vials differ most; the
low-variance channels are baseline that is nearly identical across samples and carry
almost no discriminating information.

Concretely, the top ~300 of 3401 features (≈9%) hold ~35% of the total variance, and
~440 features sit above 25% of the peak variance. This concentration means the spectra
are highly redundant and **effectively low-dimensional**: a few linear combinations of the
informative bands capture most of the variation. That is exactly the structure PCA
exploits in 3c, where we expect a small number of large "signal" eigenvalues followed by
a long noise tail.

In [ ]:
#Cell 16 — Question 3c: eigendecomposition of the covariance matrix (PCA)

# Covariance matrix  C = (1/N~) X~^T X~   (M x M = 3401 x 3401).
C = (X_centered.T @ X_centered) / N_tilde

# eigh is for SYMMETRIC matrices: it guarantees real eigenvalues and an
# orthonormal eigenvector basis, and is faster/more stable than eig.
# It returns them in ASCENDING order, so we reverse to largest-first.
w, V = np.linalg.eigh(C)
order   = np.argsort(w)[::-1]
eigvals = np.clip(w[order], 0, None)    # clip tiny negatives caused by round-off
eigvecs = V[:, order]                   # columns = principal directions nu_k (feature space)

# Sanity check: sum of eigenvalues == trace(C) == total variance.
print("sum(eigenvalues) =", round(float(eigvals.sum()), 4),
      " vs  tr(C) =", round(float(C.trace()), 4))

# 3c(i): ratio of the two largest eigenvalues
print(f"3c(i)  lambda1/lambda2 = {eigvals[0] / eigvals[1]:.3f}")

In [ ]:
#Cell 17 — Question 3c(ii)+(iii): eigenvalue spectrum (linear + log) and the elbow

# 3c(iii): locate the elbow = the largest multiplicative drop between
# consecutive eigenvalues in the leading part of the spectrum.
ratios = eigvals[:30] / eigvals[1:31]
k_star = int(np.argmax(ratios)) + 1             # number of "signal" eigenvalues
print(f"3c(iii)  elbow at k* = {k_star}  "
      f"(drop lambda{k_star}/lambda{k_star+1} = {ratios[k_star-1]:.2f})")

top = 50                                         # show the leading spectrum
k = np.arange(1, top + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, scale in zip(ax, ["linear", "log"]):
    a.plot(k, eigvals[:top], "o-", ms=4, color="steelblue")
    a.axvline(k_star + 0.5, color="red", ls="--", label=f"elbow k* = {k_star}")
    a.set_yscale(scale)
    a.set_xlabel("component index k")
    a.set_ylabel(r"eigenvalue $\lambda_k$")
    a.set_title(f"Eigenvalue spectrum ({scale} scale)")
    a.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_spectrum.png", dpi=120)
plt.show()

**3c(iii) — the elbow.** On the log-scale plot the eigenvalues fall steeply for the first
eight components and then flatten into a near-horizontal *noise floor*. The boundary is
sharp: λ₈ ≈ 0.039 but λ₉ ≈ 0.0093, a **4.1× drop** — the largest consecutive drop in the
spectrum — after which every further eigenvalue is ≈ 0.009 and decreases only marginally.
So **k\* = 8**: eight large "signal" eigenvalues followed by a long flat tail of ~900 small
"noise" eigenvalues. This is satisfyingly close to the ~10 dominant molecules named in 3h —
the spectra are mixtures of a handful of chemical components, so only a handful of
independent directions carry real signal.

**3c(iv) — what a large ratio means.** λ₁/λ₂ ≈ 3.5, and λ₁ alone explains ≈ 31% of the total
variance — far more than any other direction. A large ratio means **one direction dominates
the sample-to-sample variation**: the data has a strong principal axis and is effectively
low-dimensional, so projecting onto a few leading eigenvectors loses very little. (It also
means the Power Method of Q2 would converge quickly here, since the spectral gap λ₁ − λ₂ is
wide.)

In [ ]:
#Cell 18 — Question 3c (extra): cumulative variance explained

# Fraction of total variance captured by the top-k principal components.
cumvar = np.cumsum(eigvals) / eigvals.sum()

print(f"signal components (elbow): k* = {k_star}  ->  {100*cumvar[k_star-1]:.1f}% of variance")
for pct in [0.80, 0.90, 0.95, 0.99]:
    k_pct = int(np.searchsorted(cumvar, pct)) + 1
    print(f"{int(pct*100)}% of variance reached at k = {k_pct}")

k95 = int(np.searchsorted(cumvar, 0.95)) + 1

# Plot cumulative variance vs k, marking the elbow and the 95% point.
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(np.arange(1, len(cumvar) + 1), cumvar, color="steelblue")
ax.axhline(0.95, color="gray", ls=":", lw=1)
ax.axvline(k_star, color="red",   ls="--", lw=1,
           label=f"elbow k* = {k_star}  ({100*cumvar[k_star-1]:.0f}% of variance)")
ax.axvline(k95,    color="green", ls="--", lw=1,
           label=f"95% variance at k = {k95}")
ax.set_xlabel("number of components k")
ax.set_ylabel("cumulative variance explained")
ax.set_title("Cumulative variance: 8 signal components reach only ~55%; 95% needs ~670")
ax.set_ylim(0, 1.02)
ax.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_cvarr.png", dpi=120)
plt.show()

In [ ]:
#Cell 20 — Question 3c: table of PCA eigenvalue ratios (saved to the q3c outputs)

# For each leading component k: the eigenvalue, the consecutive ratio
# lambda_k / lambda_{k+1} (whose largest value marks the elbow), and the
# per-component and cumulative share of the total variance.
K = 12
tot = eigvals.sum()
rows, ratios_list = [], []
for k in range(K):
    lam = eigvals[k]
    ratio = eigvals[k] / eigvals[k + 1]
    ratios_list.append(ratio)
    rows.append([str(k + 1), f"{lam:.4f}", f"{ratio:.2f}",
                 f"{100 * lam / tot:.1f}%", f"{100 * eigvals[:k + 1].sum() / tot:.1f}%"])
elbow = int(np.argmax(ratios_list))   # the largest consecutive drop = the elbow

fig, ax = plt.subplots(figsize=(8, 4.8)); ax.axis("off")
tbl = ax.table(cellText=rows,
               colLabels=["k", "eigenvalue \u03bbk", "ratio \u03bbk/\u03bbk+1",
                          "variance %", "cumulative %"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.5)
for j in range(5):
    tbl[elbow + 1, j].set_facecolor("#ffe08a")   # highlight the elbow row
ax.set_title(f"PCA eigenvalue ratios — largest drop \u03bb{elbow+1}/\u03bb{elbow+2} = "
             f"{ratios_list[elbow]:.2f} marks the elbow k* = {elbow+1}")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_ratios.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"PCA eigenvalue ratios (top {K}):")
print(f"{'k':>2}  {'lambda':>9}  {'ratio':>6}  {'var%':>6}  {'cum%':>6}")
for r in rows:
    print(f"{r[0]:>2}  {r[1]:>9}  {r[2]:>6}  {r[3]:>6}  {r[4]:>6}")

In [ ]:
#Cell 22 — Question 3c: save the eigenvalue-ratio table (k = 1..9) as a loadable CSV
import csv
tot = eigvals.sum()
K = 9
os.makedirs("data/output/TASK 3", exist_ok=True)
csv_path = "data/output/TASK 3/q3c_ratios.csv"
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["k", "eigenvalue", "ratio_k_over_k+1", "variance_percent", "cumulative_percent"])
    for k in range(K):
        wr.writerow([k + 1,
                     round(float(eigvals[k]), 6),
                     round(float(eigvals[k] / eigvals[k + 1]), 4),
                     round(float(100 * eigvals[k] / tot), 3),
                     round(float(100 * eigvals[:k + 1].sum() / tot), 3)])
print("saved", csv_path)

# verify it loads back
with open(csv_path, newline="") as f:
    for row in csv.reader(f):
        print(row)

**Cumulative variance — why the elbow, not the "95%" rule, sets the dimensionality.**
The 8 signal components (the elbow) capture only ≈ 55.5% of the total variance; reaching 95%
requires **k = 667** components. That gap exists because the variance beyond the elbow is
spread thinly over hundreds of small *noise* eigenvalues, which add up collectively even
though none is individually meaningful. So a "keep 95% of the variance" cut-off would pull in
~660 noise directions and badly **over**estimate the data's dimensionality. The elbow /
spectral-gap criterion (k\* = 8) is the right one here, and it matches the ~10 dominant
molecules of 3h.

In [ ]:
#Cell 19 — Question 3d(i): plot the leading eigenvector v1

v1 = eigvecs[:, 0].copy()
# eigenvectors are defined up to a sign; flip so the largest component is positive
if v1[np.argmax(np.abs(v1))] < 0:
    v1 = -v1

# which frequencies carry the most weight in v1?
top = np.argsort(np.abs(v1))[::-1][:10]
print("most heavily weighted features (j, v1_j):",
      [(int(j), round(float(v1[j]), 3)) for j in top])
print("fraction of negative components:", round(float((v1 < 0).mean()), 3))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v1, lw=0.6, color="darkgreen")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("feature index j  (frequency channel)")
ax.set_ylabel(r"$\nu_1$ component")
ax.set_title(r"Leading eigenvector $\nu_1$  (direction of greatest variation)")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3d_v1.png", dpi=120)
plt.show()

**3d(i) — interpreting ν₁.** The leading eigenvector is *not* flat: its weight is concentrated
in two narrow frequency bands — around **j ≈ 900–960** (the largest components: j = 910, 922,
942, 909, …) and **j ≈ 3150–3260** (j = 3150, 3254). These are exactly the high-variance
molecular bands seen in 3b. ν₁ also has both **positive and negative lobes** (~27 % of its
components are negative), so it is a *contrast* direction: it separates samples by the relative
balance of absorption between these bands, not by overall brightness. The single summary number
z = x·ν₁ therefore measures "how strongly a vial absorbs in the 900–960 / 3150–3260 bands versus
the rest" — the chemical contrast that most distinguishes the concoctions.

In [ ]:
#Cell 21 — Question 3d(ii): does v1 resemble a measurement?

def cosines(M, v):                      # cosine similarity of v with every row of M
    return (M @ v) / (np.linalg.norm(M, axis=1) * np.linalg.norm(v))

cos_centered = cosines(X_centered, v1)              # vs mean-centered rows (rows of X~)
i_best = int(np.argmax(np.abs(cos_centered)))
mean_spec = S_clean.mean(0)
print(f"best match: centered row #{i_best}, cos = {cos_centered[i_best]:.3f}")
print(f"cos(v1, raw row #{i_best})  = {cosines(S_clean, v1)[i_best]:.3f}")
print(f"cos(v1, mean spectrum)     = "
      f"{(v1 @ mean_spec) / (np.linalg.norm(v1) * np.linalg.norm(mean_spec)):.3f}")

# overlay v1 with the best-matching centered measurement (both unit-normalised)
row = X_centered[i_best] / np.linalg.norm(X_centered[i_best])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v1,  lw=0.6, color="darkgreen", label=r"$\nu_1$")
ax.plot(row, lw=0.6, color="orange", alpha=0.8,
        label=f"centered measurement #{i_best} (unit norm)")
ax.set_xlabel("feature index j"); ax.set_ylabel("amplitude")
ax.set_title(rf"$\nu_1$ vs best-matching centered spectrum  (cos = {cos_centered[i_best]:.2f})")
ax.legend()
plt.tight_layout()
plt.savefig("data/output/TASK 3/q3d_v1_vs_row.png", dpi=120)
plt.show()

**3d(ii) — does ν₁ resemble a measurement?** Yes — but a *mean-centered* one. ν₁ matches the
centered spectrum of row #193 with **cosine ≈ 0.92** (overlay above), so the dominant axis of
variation is essentially the deviation-from-average signature of one particular concoction. By
contrast it matches the *raw* (un-centered) spectra only moderately (cos ≈ 0.67) and is
essentially **orthogonal to the mean spectrum** (cos ≈ 0.03). That is exactly what mean-centering
(3b) buys us: the raw spectra are dominated by the shared average spectrum, but ν₁ — living in the
centered space by construction — ignores that common part and captures how the most-deviated
concoction differs from the mean. So ν₁ looks like a real *difference*-spectrum of the data, not
like the average spectrum every vial shares.

In [ ]:
#Cell 23 — Question 3e(i): project onto the top 4 PCs and scatter all 6 pairs

# Z = X~ V4 : each cleaned vial summarised by its scores on the 4 leading
# eigenvectors (a 4-D compressed coordinate). Shape (924, 4).
Z = X_centered @ eigvecs[:, :4]

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, (p, q) in zip(ax.ravel(), pairs):
    a.scatter(Z[:, p], Z[:, q], s=10, alpha=0.4)      # small, semi-transparent points
    a.set_xlabel(f"PC{p+1}"); a.set_ylabel(f"PC{q+1}")
fig.suptitle("Covariance-space projection: all 6 pairwise scatter plots of the top-4 PCs")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3e_covariance_scatter.png", dpi=110)
plt.show()

In [ ]:
#Cell 24 — Question 3e(ii): how many distinct clusters?

def count_clusters(P, r):
    # leader clustering (numpy-only): new cluster when a point is farther than r
    # from every existing cluster centre
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

D = np.sqrt(((Z[:, None] - Z[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
r = 8 * np.median(D.min(1))                 # radius ~ 8x the typical neighbour spacing
print(f"distinct clusters (covariance space): {count_clusters(Z, r)}")

**3e(ii) — counting the clusters.** The six panels show many tight, well-separated blobs. A simple
distance-based count (leader clustering, stable for a radius of 6–12× the typical neighbour spacing)
gives **25 clusters**. Each blob is a group of vials with almost identical spectra — i.e. one
**concoction**. So the 924 cleaned vials fall into about 25 distinct recipes, comfortably under the
"fewer than 100 unique samples" stated in 3h. The clean separation into a small number of groups is
the visible payoff of the low-dimensional structure found in 3c: projecting onto just 4 of 3401
directions is enough to resolve the recipes.

In [ ]:
#Cell 25 — Question 3f(i)+(ii): cluster via the Gram matrix G = (1/N~) X~ X~^T

# The Gram matrix lives in SAMPLE space (N~ x N~); entry G_ij is the similarity
# between vials i and j. It shares the nonzero eigenvalues of the covariance matrix C.
G = (X_centered @ X_centered.T) / N_tilde
wG, UG = np.linalg.eigh(G)
order = np.argsort(wG)[::-1]
gvals = np.clip(wG[order], 0, None)
gvecs = UG[:, order]                       # columns: N~-dimensional sample-space eigenvectors

print("Gram eigenvalues (top 8):", np.round(gvals[:8], 4))
print("match covariance eigenvalues:", np.allclose(gvals[:8], eigvals[:8], atol=1e-6))

# A Gram eigenvector's components ARE the sample coordinates (no projection of X~
# needed). We scale by sqrt(N~ * lambda) so the units match the covariance-space
# scores of 3e, i.e. U*sqrt(N~ lambda) = X~ V.
Zg = gvecs[:, :4] * np.sqrt(N_tilde * gvals[:4])

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, (p, q) in zip(ax.ravel(), pairs):
    a.scatter(Zg[:, p], Zg[:, q], s=10, alpha=0.4, color="seagreen")
    a.set_xlabel(f"Gram PC{p+1}"); a.set_ylabel(f"Gram PC{q+1}")
fig.suptitle("Gram-space projection: 6 pairwise scatter plots of the top-4 Gram eigenvectors")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3f_gram_scatter.png", dpi=110)
plt.show()

In [ ]:
#Cell 26 — Question 3f(iii): how many clusters in Gram space?

def count_clusters(P, r):
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

D = np.sqrt(((Zg[:, None] - Zg[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
r = 8 * np.median(D.min(1))                 # radius ~ 8x the typical neighbour spacing
print(f"distinct clusters (Gram space): {count_clusters(Zg, r)}")

**3f(iii) — clusters via the Gram matrix.** The Gram matrix \(G = \tfrac1{\tilde N}\tilde X\tilde X^{\top}\) lives in *sample* space (924\u00d7924); its eigenvectors are already \(\tilde N\)-dimensional, so each component is directly the coordinate of one vial — no projection of \(\tilde X\) is needed. \(G\) shares the nonzero eigenvalues of the covariance matrix \(C\) (verified above), exactly as Question 2d predicts. Counting with the same leader-clustering rule gives **25 clusters** — identical to 3e and stable across radii 6\u201312\u00d7 the median neighbour spacing. The six Gram-space scatter plots reproduce those of 3e up to per-axis sign flips (eigenvectors are defined only up to a sign): the two matrices are two views of the same structure. The detailed comparison is the subject of 3g.

In [ ]:
#Cell 27 — Question 3g(i): are the 3e (covariance) and 3f (Gram) clusterings the same?

# Both 3e and 3f returned 25 clusters. The reason is that the two coordinate sets are the
# SAME matrix up to a per-axis sign: via the SVD X~ = U S V^T, the covariance scores are
# Z = X~ V = U S, while the Gram coordinates are Zg = U * sqrt(N~ * lambda) = U S. We confirm
# this by correlating the 3e scores Z with the 3f scores Zg, component by component.
def _corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

M = np.array([[_corr(Z[:, i], Zg[:, j]) for j in range(4)] for i in range(4)])
np.set_printoptions(precision=3, suppress=True)
print("correlation matrix (rows = 3e PCs, cols = 3f PCs):")
print(M)
print("per-matching-component |correlation|:", np.round(np.abs(np.diag(M)), 4))
print("max off-diagonal |correlation|      :", round(float(np.abs(M - np.diag(np.diag(M))).max()), 4))
print("coordinates equal up to per-axis sign?", bool(np.allclose(np.abs(Z), np.abs(Zg), atol=1e-6)))

## Question 3g — comparing the covariance and Gram clusterings

**3g(i) — Do we find the same number of clusters? Yes (25 in both), and necessarily so.**

*The mathematical connection.* If \(\nu\) is an eigenvector of \(C=\tfrac1{\tilde N}\tilde X^{\top}\tilde X\) with eigenvalue \(\lambda\), then \(\tilde X\nu\) is an eigenvector of \(G=\tfrac1{\tilde N}\tilde X\tilde X^{\top}\) with the **same** eigenvalue \(\lambda\) (substitute and check). So \(C\) and \(G\) share their nonzero eigenvalues — which the notebook verified exactly (top‑8 identical).

More strongly, write the SVD \(\tilde X = U\Sigma V^{\top}\). The covariance eigenvectors are the columns of \(V\) and the covariance‑space scores are \(Z = \tilde X V = U\Sigma\); the Gram eigenvectors are the columns of \(U\) and, scaled by \(\sqrt{\tilde N\lambda}=\sigma\), the Gram coordinates are \(Z_g = U\sigma = U\Sigma\). **Both coordinate sets are the same matrix \(U\Sigma\)** — identical up to the arbitrary sign of each eigenvector. The cell above confirms it: every matching component is perfectly correlated (\(|\mathrm{corr}|=1.000\)), all cross‑correlations are \(0\), and \(|Z| = |Z_g|\) to machine precision (only PC1 and PC4 happen to come out sign‑flipped here). Because the two point clouds are the same up to reflecting individual axes, the clusters — and therefore their count — must be identical. (In general the two pictures can differ slightly when one matrix is better conditioned numerically, but here the structure is clean enough that they agree exactly.)

**3g(ii) — What it means for a vial to belong to a cluster, in each space.**

*Covariance space (3e).* A vial's coordinates are the projections of its centred spectrum onto the top‑\(k\) **feature‑space** eigenvectors, \(z_i=(\tilde x_i\cdot\nu_1,\dots,\tilde x_i\cdot\nu_k)\). Each coordinate measures how strongly the vial expresses one dominant *pattern of spectral variation* — a fixed weighted combination of frequencies (e.g. \(\nu_1\)'s contrast between molecular bands). Two vials are close when they express these dominant chemical patterns to the same degree, i.e. their spectra are similar mixtures along the directions in which samples differ most. A cluster is a set of vials sharing that fingerprint.

*Gram space (3f).* A vial's coordinates are the components of the top‑\(k\) **sample‑space** eigenvectors of \(G\). Since \(G_{ij}\) is the dot‑product similarity between vials \(i\) and \(j\), each Gram eigenvector is a pattern defined over the 924 vials, and a vial's coordinate is its loading in that pattern — a compressed summary of *how similar this vial is to every other vial*. Two vials are close when they have the same similarity‑fingerprint to the rest of the dataset.

*Same conclusion, two viewpoints.* Covariance space asks "**which mixture of frequencies** does this vial express?"; Gram space asks "**which other vials** does this one resemble?" Both place vials with near‑identical spectra at the same point, so proximity in either coordinate system means the underlying vials hold the **same chemical concoction** — which is exactly why the two routes agree on 25 clusters.